In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Sidebar switching is state-only (verified: no router in StaffApp.jsx),
    # so the URL stays put while the visible module changes.
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Customers')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'CRM')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='CRM sections']")))
    print("At CRM, URL:", driver.current_url)

    # Back() walks real browser history: dashboard page -> /login (login POST used assign).
    # With a valid token App.jsx still renders the staff app (verified in App.jsx).
    driver.back()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    assert not driver.find_elements(By.ID, "username"), "Session lost after Browser Back."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token, "Access token missing after Browser Back."
    print("After Back, URL:", driver.current_url)
    print("Staff shell still rendered, session intact.")
    print("PASS: Browser Back navigation works")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("44_browser_back_FAIL.png")
finally:
    driver.quit()